# Lenguajes y Autómatas II (SCD-1016)
## Sesión 17 · Comprobación Semántica de Retornos y Rutas de Control

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo sabe el compilador que una función siempre devuelve un valor, incluso cuando toma un camino con if?
>
> **Objetivo de la sesión:** Verificar el tipo de cada return y que todas las rutas de una función no-void terminen en un retorno, devolviendo un diagnóstico único.

**Entregable:** tu función diagnosticar_retorno() en artesano_sesion17_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion17.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes el tipo que promete la función, el tipo de la expresión devuelta y el conteo de rutas totales y rutas que sí retornan. Debes emitir 0 si todo está bien, 1 si el tipo es incompatible y 2 si queda una ruta sin retorno.

Al final de la sesión tendrás: **Un diagnóstico de retornos que distingue tipo incompatible, ruta sin retorno y función correcta.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 17](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion17.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion17.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion17.h"

int diagnosticar_retorno(const char *tipo_funcion, const char *tipo_expresion, int rutas_totales, int rutas_con_return);
```

- La conversión solo se acepta cuando el tipo destino es igual o más ancho que el origen.
- Devolver un valor en una función void es siempre un error de tipo.
- Una firma NULL o un conteo de rutas incoherente (cero rutas o más retornos que rutas) devuelve -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion17_c.c` junto con `artesano_sesion17_c.c`
> (o `ia_sesion17_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `int con int y dos rutas` | 0 |
| 2 | `void devuelve valor` | 1 |
| 3 | `ruta sin retorno` | 2 |
| 4 | `double acepta int` | 0 |
| 5 | `int no acepta double` | 1 |
| 6 | `cero rutas de ejecucion` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 17 - Validacion de retornos y rutas ===
Caso                               esperado:              obtenido:              resultado
[ 1] int con int y dos rutas            esperado: 0            obtenido: 0            OK
[ 2] void devuelve valor                esperado: 1            obtenido: 1            OK
[ 3] ruta sin retorno                   esperado: 2            obtenido: 2            OK
[ 4] double acepta int                  esperado: 0            obtenido: 0            OK
[ 5] int no acepta double               esperado: 1            obtenido: 1            OK
[ 6] cero rutas de ejecucion            esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion17.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El tipo se compara antes que las rutas: un return incompatible nunca se reporta como ruta faltante.
- [ ] Una función void solo es válida cuando la expresión devuelta también es void.
- [ ] El conteo de rutas detecta exactamente el caso de un if sin else que deja un camino abierto.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion17_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion17.h
/* api_sesion17.h - contrato de la Sesion 17
 * Validación de retornos y rutas
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION17_H
#define API_SESION17_H

int diagnosticar_retorno(const char *tipo_funcion, const char *tipo_expresion, int rutas_totales, int rutas_con_return);

#endif /* API_SESION17_H */


In [ ]:
%%writefile artesano_sesion17_c.c
/* artesano_sesion17_c.c - Fase 1: El Artesano (sin IA)
 * Validación de retornos y rutas.
 *
 * Implementa lo que declara api_sesion17.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion17.h"
#include <stdio.h>
#include <string.h>

int diagnosticar_retorno(const char *tipo_funcion, const char *tipo_expresion, int rutas_totales, int rutas_con_return) {
    (void)tipo_funcion; (void)tipo_expresion; (void)rutas_totales; (void)rutas_con_return; /* TODO */
    return -1;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion17_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion17_c.c
/* ia_sesion17_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion17.h. No puede traer main().
 */
#include "api_sesion17.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion17_c.c
/* harness_sesion17_c.c - auditoria automatica de la Sesion 17
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion17_c.c artesano_sesion17_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion17.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 17 - Validacion de retornos y rutas ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = diagnosticar_retorno("int", "int", 2, 2); verificar_int("int con int y dos rutas", (long)(0), (long)(r)); }
    { int r = diagnosticar_retorno("void", "int", 1, 1); verificar_int("void devuelve valor", (long)(1), (long)(r)); }
    { int r = diagnosticar_retorno("int", "int", 2, 1); verificar_int("ruta sin retorno", (long)(2), (long)(r)); }
    { int r = diagnosticar_retorno("double", "int", 1, 1); verificar_int("double acepta int", (long)(0), (long)(r)); }
    { int r = diagnosticar_retorno("int", "double", 1, 1); verificar_int("int no acepta double", (long)(1), (long)(r)); }
    { int r = diagnosticar_retorno("int", "int", 0, 0); verificar_int("cero rutas de ejecucion", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion17.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion17_c.c artesano_sesion17_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion17_c.c ia_sesion17_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion17.md` a `auditorias/auditoria_sesion17.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
